# 32 — Slim training file + basal means (CPU)

* `data/pbmc_train90_slim.h5ad`: obs, `obsm['X_pca']`, `uns` — **no expression matrix** (`X` is empty). Every GPU run loads this (seconds, not minutes);
  the expression matrix in `pbmc_train90.h5ad` is only needed for gene-space metrics.
* `data/basal_means_train90.npz`: mean log1p(CP10k) of PBS cells per (donor, cell_type) over the 2,000 HVGs — the input of the `mlp_full_basal` gate.

In [1]:
import numpy as np, pandas as pd, anndata as ad, scipy.sparse as sp, time
t0=time.time(); a=ad.read_h5ad("../data/pbmc_train90.h5ad"); print(a, "%.0fs"%(time.time()-t0))
o=a.obs.copy()
for c in ["donor","cell_type","cytokine"]: o[c]=o[c].astype(str)
pbs=np.where(o.is_control.values)[0]; X=a.X[pbs].toarray() if sp.issparse(a.X) else a.X[pbs]
key=(o.donor+"|"+o.cell_type).values[pbs]; ks=sorted(set(key)); M=np.stack([X[key==k].mean(0) for k in ks]).astype(np.float32)
np.savez("../data/basal_means_train90.npz",keys=np.array(ks),X=M,genes=np.array(a.var_names)); print("basal means",M.shape)
s=ad.AnnData(X=sp.csr_matrix((a.n_obs,1),dtype=np.float32),obs=a.obs.copy(),obsm={"X_pca":a.obsm["X_pca"].copy()},uns=a.uns)
s.write_h5ad("../data/pbmc_train90_slim.h5ad"); print(s)

AnnData object with n_obs × n_vars = 3411214 × 2000
    obs: 'sample', 'species', 'gene_count', 'tscp_count', 'mread_count', 'bc1_wind', 'bc2_wind', 'bc3_wind', 'bc1_well', 'bc2_well', 'bc3_well', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'total_counts_MT', 'pct_counts_MT', 'log1p_total_counts_MT', 'donor', 'cytokine', 'treatment', 'cell_type', 'cytokine_family', 'is_control', 'condition'
    var: 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'donor_embeddings', 'esm2_embeddings', 'pca', 'recipe'
    obsm: 'X_pca'
    layers: 'counts' 40s


basal means (72, 2000)


AnnData object with n_obs × n_vars = 3411214 × 1
    obs: 'sample', 'species', 'gene_count', 'tscp_count', 'mread_count', 'bc1_wind', 'bc2_wind', 'bc3_wind', 'bc1_well', 'bc2_well', 'bc3_well', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'total_counts_MT', 'pct_counts_MT', 'log1p_total_counts_MT', 'donor', 'cytokine', 'treatment', 'cell_type', 'cytokine_family', 'is_control', 'condition'
    uns: 'donor_embeddings', 'esm2_embeddings', 'pca', 'recipe'
    obsm: 'X_pca'


In [2]:
print(o.cytokine.nunique(),"cytokines (incl PBS);",o.donor.nunique(),"donors;",a.n_obs,"cells")
print(o[~o.is_control.values].groupby(["donor"]).size())
print(sorted(o.cell_type.unique()))

91 cytokines (incl PBS); 4 donors; 3411214 cells


donor
Donor1     857921
Donor12    874956
Donor2     813975
Donor8     589147
dtype: int64
['B Intermediate/Memory', 'B Naive', 'CD14 Mono', 'CD16 Mono', 'CD4 Memory', 'CD4 Naive', 'CD8 Memory', 'CD8 Naive', 'HSPC', 'ILC', 'MAIT', 'NK', 'NK CD56bright', 'NKT', 'Plasmablast', 'Treg', 'cDC', 'pDC']
